<a href="https://colab.research.google.com/github/rodrigorissettoterra/Brazil_B3_Stocks_Analysis/blob/main/An%C3%A1lise_Fundamentalista_e_T%C3%A9cnica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Brazil B3 Stocks Analysis (Data)

Google Sheets with 2 tabs:
- Tab 1: Dados (Fundamentus + YFinance (raw): Merged)
- Tab 2: Normalizados (Normalized data (0 to 1): Merged)

Notes:
- Uses Fundamentus as the universe (filters price>0 and liquidity>0)
- Computes technical snapshot (2y daily history) ONLY for those tickers
- Drops tickers with no technical data
- Select ONLY the columns we will use for analysis

In [ ]:
!pip -q install fundamentus yfinance gspread pandas numpy

from google.colab import auth
auth.authenticate_user()

import contextlib
import io
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import fundamentus
import yfinance as yf
import gspread
from google.auth import default

## Google Sheets auth

In [ ]:
creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1GyMu7RiKHfbDA0Bio8VG8LjmX2aHlfy6BdE1GwRq4o8/"
TAB_DADOS = "Dados"
TAB_NORM  = "Normalizados"

## Selected columns

Fundamental columns we will use

In [ ]:
FUND_COLS_FINAL = [
    "ticker",
    "price",
    "dividend_yield",
    "roic",
    "roe",
    "net_margin",
    "revenue_growth_5y",
    "debt_to_equity",
    "current_ratio",
    "avg_daily_liquidity_2m",
]

Technical columns we will use

In [ ]:
TECH_COLS_FINAL = [
    "rsi_14",
    "dist_sma_200",
    "macd_hist",
    "ret_20d",
    "vol_rel",
]

Final column order (Fundamental + Technical)

In [ ]:
FINAL_COL_ORDER = FUND_COLS_FINAL + TECH_COLS_FINAL

## Helpers

### Robust float conversion

In [ ]:
def robust_to_float(series: pd.Series) -> pd.Series:
    x = series.copy()
    if pd.api.types.is_numeric_dtype(x):
        return pd.to_numeric(x, errors="coerce")

    x = x.astype(str).str.strip()
    x = x.replace({"": np.nan, "None": np.nan, "nan": np.nan})
    x = x.str.replace("%", "", regex=False)

    def _parse_one(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return np.nan
        v = str(v).strip()
        if v == "" or v.lower() == "nan":
            return np.nan

        has_comma = "," in v
        has_dot = "." in v

        # Decide decimal separator by the last punctuation position
        if has_comma and has_dot:
            if v.rfind(",") > v.rfind("."):
                v = v.replace(".", "").replace(",", ".")   # 1.234,56
            else:
                v = v.replace(",", "")                     # 1,234.56
        elif has_comma and not has_dot:
            v = v.replace(",", ".")                        # 1234,56

        try:
            return float(v)
        except:
            return np.nan

    return x.map(_parse_one)

### Sheets range + write without wiping extra columns

In [ ]:
def col_to_a1(n: int) -> str:
    s = ""
    while n > 0:
        n, r = divmod(n - 1, 26)
        s = chr(65 + r) + s
    return s

def ensure_worksheet(sh, title: str, rows: int = 3000, cols: int = 30):
    try:
        return sh.worksheet(title)
    except gspread.WorksheetNotFound:
        return sh.add_worksheet(title=title, rows=rows, cols=cols)

def clear_and_write_df(ws, df: pd.DataFrame):
    nrows = len(df) + 1
    ncols = len(df.columns)
    end_col = col_to_a1(ncols)
    rng = f"A1:{end_col}{nrows}"
    ws.batch_clear([rng])
    values = [df.columns.tolist()] + df.replace({np.nan: ""}).values.tolist()
    ws.update(rng, values, value_input_option="USER_ENTERED")
    return rng

### Technical indicators

In [ ]:
def sma(s, n): return s.rolling(n).mean()
def ema(s, n): return s.ewm(span=n, adjust=False).mean()

def rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def macd(close: pd.Series, fast=12, slow=26, signal=9):
    macd_line = ema(close, fast) - ema(close, slow)
    signal_line = ema(macd_line, signal)
    hist = macd_line - signal_line
    return macd_line, signal_line, hist

def yf_download_silent(symbol: str, period: str, interval: str) -> pd.DataFrame:
    f = io.StringIO()
    with contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
        dfp = yf.download(symbol, period=period, interval=interval, progress=False, auto_adjust=False)
    return dfp

### Get Fundamentus + map columns to English

In [ ]:
df = fundamentus.get_resultado().reset_index()

col_map_pt_en = {
    "papel": "ticker",
    "cotacao": "price",
    "dy": "dividend_yield",
    "roic": "roic",
    "roe": "roe",
    "mrgliq": "net_margin",
    "c5y": "revenue_growth_5y",
    "divbpatr": "debt_to_equity",
    "liqc": "current_ratio",
    "liq2m": "avg_daily_liquidity_2m",
}
df = df.rename(columns={c: col_map_pt_en[c] for c in df.columns if c in col_map_pt_en})

if "ticker" not in df.columns:
    df = df.rename(columns={df.columns[0]: "ticker"})

### Convert numeric columns robustly

In [ ]:
for c in df.columns:
    if c == "ticker":
        continue
    conv = robust_to_float(df[c])
    if conv.notna().sum() > 0:
        df[c] = conv

### Keep ONLY the fundamental columns we use

In [ ]:
for c in FUND_COLS_FINAL:
    if c not in df.columns:
        df[c] = np.nan
df_fund = df[FUND_COLS_FINAL].copy()

### Filters: price > 0, liquidity > 0

In [ ]:
df_fund = df_fund[df_fund["price"].notna() & (df_fund["price"] > 0)].copy()
df_fund = df_fund[df_fund["avg_daily_liquidity_2m"].notna() & (df_fund["avg_daily_liquidity_2m"] > 0)].copy()

## Technical snapshot for SAME tickers

In [ ]:
PERIOD = "2y"
INTERVAL = "1d"

SMA_LONG = 200
RSI_PERIOD = 14

tech_rows = []

tickers = df_fund["ticker"].astype(str).str.strip().tolist()
tickers_yf = [t + ".SA" if not t.endswith(".SA") else t for t in tickers]

for t_b3, t_yf in zip(tickers, tickers_yf):
    dfp = yf_download_silent(t_yf, PERIOD, INTERVAL)
    if dfp is None or dfp.empty:
        continue

    # Normalize columns (handle possible multiindex)
    if isinstance(dfp.columns, pd.MultiIndex):
        dfp.columns = dfp.columns.get_level_values(0)
    dfp.columns = [str(c).capitalize() for c in dfp.columns]

    required = {"Open", "High", "Low", "Close", "Volume"}
    if not required.issubset(set(dfp.columns)):
        continue

    dfp = dfp.dropna(subset=list(required)).copy()
    if dfp.empty:
        continue

    # Indicators we need
    dfp["sma_200"] = sma(dfp["Close"], SMA_LONG)
    dfp["rsi_14"] = rsi(dfp["Close"], RSI_PERIOD)
    _, _, macd_hist = macd(dfp["Close"])
    dfp["macd_hist"] = macd_hist
    dfp["ret_20d"] = dfp["Close"].pct_change(20)
    dfp["vol_avg_20"] = dfp["Volume"].rolling(20).mean()
    dfp["vol_rel"] = dfp["Volume"] / dfp["vol_avg_20"]
    dfp["dist_sma_200"] = (dfp["Close"] / dfp["sma_200"]) - 1

    last = dfp.iloc[-1]

    row = {
        "ticker": t_b3,
        "rsi_14": float(last.get("rsi_14", np.nan)),
        "dist_sma_200": float(last.get("dist_sma_200", np.nan)),
        "macd_hist": float(last.get("macd_hist", np.nan)),
        "ret_20d": float(last.get("ret_20d", np.nan)),
        "vol_rel": float(last.get("vol_rel", np.nan)),
    }

    # If everything is NaN, skip
    if all(pd.isna(row[k]) for k in TECH_COLS_FINAL):
        continue

    tech_rows.append(row)

df_tech = pd.DataFrame(tech_rows)

# If no technical data, stop early (avoid exporting empties)
if df_tech.empty:
    raise RuntimeError("No technical data could be fetched from yfinance for the filtered universe.")

## Merge FUND + TECH and keep ONLY FINAL columns

In [ ]:
df_final = df_fund.merge(df_tech, on="ticker", how="inner").copy()
df_final = df_final[FINAL_COL_ORDER].copy()

## Create Normalized table

In [ ]:
INCLUDE_RSI_SCORE = True

df_norm = df_final.copy()

numeric_cols = [c for c in df_norm.columns if c != "ticker"]
for c in numeric_cols:
    df_norm[c] = pd.to_numeric(df_norm[c], errors="coerce")

### Max-normalization

In [ ]:
col_max = df_norm[numeric_cols].max(skipna=True).replace({0: np.nan})
df_norm[numeric_cols] = df_norm[numeric_cols].div(col_max, axis=1)

if "debt_to_equity" in df_norm.columns:
    df_norm["debt_to_equity"] = 1 - df_norm["debt_to_equity"]

if INCLUDE_RSI_SCORE:
    rsi_raw = df_final["rsi_14"].astype(float)
    rsi_score = 1 - (rsi_raw.sub(50).abs() / 50)
    rsi_score = rsi_score.clip(lower=0, upper=1)
    df_norm["rsi_score"] = rsi_score
    cols = df_norm.columns.tolist()
    if "rsi_score" in cols:
        cols.remove("rsi_score")
        idx = cols.index("rsi_14") + 1 if "rsi_14" in cols else len(cols)
        cols.insert(idx, "rsi_score")
        df_norm = df_norm[cols]

## Export to Google Sheets

In [ ]:
sh = gc.open_by_url(SPREADSHEET_URL)

ws_dados = ensure_worksheet(sh, TAB_DADOS, rows=max(3000, len(df_final)+10), cols=max(30, len(df_final.columns)+5))
ws_norm  = ensure_worksheet(sh, TAB_NORM,  rows=max(3000, len(df_norm)+10),  cols=max(30, len(df_norm.columns)+5))

rng1 = clear_and_write_df(ws_dados, df_final)
rng2 = clear_and_write_df(ws_norm, df_norm)

## Final Message

In [ ]:
print(f"OK! Exported {len(df_final)} tickers.")
print(f"'{TAB_DADOS}' range: {rng1}")
print(f"'{TAB_NORM}'  range: {rng2}")
print("Done.")